In [26]:
from dfbr.utils.files import get_config
from dfbr.data.dataset import BikeDemandDataset, BikeOptTargetsDataset
from dfbr.models.station_targets import BikeStationTargets
from dfbr.models.cost_head import CostHead
from dfbr.models.mlp import MLP
from dfbr.eval.simmulation import Sim, create_station_dict, create_event_df
from dfbr.training.train import train_one_epoch, evaluate
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch.nn as nn
import pyepo
import torch
import numpy as np

#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Setup
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Read config
config = get_config("baseline.yaml")

#Create a dictionary of stations
station_dict = create_station_dict(config["paths"]["stations"], config["paths"]["station_dist_miles"], config["sim"]["start_inv_pct"])
#Sort by id to ensure alignment
station_ids = sorted(station_dict.keys())
#Get parameters for shapes of datasets and models
num_stations = len(station_dict)
capacities = [station_dict[sid].capacity for sid in station_ids]
max_cap = max(capacities)


In [2]:
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Load Datasets
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Create datasets
train_ds = BikeDemandDataset(
        file = config["paths"]["input"],
        start_date = config["data"]["train_start_date"],
        end_date = config["data"]["train_end_date"],
        target_cols= [str(id) for id in station_ids],
        input_scale_cols= ['mean_temp', 'precip', 'max_gust'],
        input_no_scale_cols=['sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month'],
        capacities=capacities,
        max_cap=max_cap
    )

training_stats = {'mean': train_ds.mean, 'std': train_ds.std, 'y_mean': train_ds.y_mean, 'y_std': train_ds.y_std}

test_ds = BikeDemandDataset(
        file = config["paths"]["input"],
        start_date = config["data"]["test_start_date"],
        end_date = config["data"]["test_end_date"],
        target_cols= [str(id) for id in station_ids],
        input_scale_cols= ['mean_temp', 'precip', 'max_gust'],
        input_no_scale_cols=['sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month'],
        capacities=capacities,
        max_cap=max_cap,
        is_train=False,
        scaling_factor=training_stats
    )

In [3]:
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Solve for optimal values with ground truth data
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
input_size = len(train_ds[0][0])
output_size = num_stations
opt_model = BikeStationTargets(num_stations=num_stations, max_cap=max_cap, total_inventory=int(sum(capacities) * config["sim"]["start_inv_pct"]))
pred_model = MLP(input_size, output_size, config["model"]["hidden_layers"])
#Create cost head
cost_head = CostHead(capacities=capacities, max_cap=max_cap)


Set parameter Username
Set parameter LicenseID to value 2774727
Academic license - for non-commercial use only - expires 2027-02-03


In [10]:
x, y, _ = train_ds[0]
with torch.no_grad():
    yp = pred_model(x.unsqueeze(0))
    cp = cost_head(yp)
opt_model.setObj(cp.squeeze(0))
wp, _ = opt_model.solve()

In [19]:
wp = np.array(wp)
wp = wp.reshape(num_stations, max_cap+1)

In [ ]:
targets = np.argmax(wp, axis =1)


535